https://liquipedia.net/commons/Liquipedia:API_Usage_Guidelines

https://liquipedia.net/leagueoflegends/index.php?title=Category:Players

and some help from ChatGPT

(this made me think of the Futurama ep "The Problem with Popplers" where the only two company names that haven't been trademarked are "Popplers" and "Zitzels" - see https://tvtropes.org/pmwiki/pmwiki.php/Recap/FuturamaS2E15TheProblemWithPopplers under Disney Owns This Trope)

In [2]:
import requests

In [9]:
headers = {
    "User-Agent": "EsportUsernames/1.0 (https://github.com/hollowscene)"
}

In [12]:
def get_player_pages(wiki: str) -> list:
    """Retrieve all player pages from a specific game's Liquidipedia."""
    url = f"https://liquipedia.net/{wiki}/api.php"

    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": "Category:Players",
        "cmlimit": "max",  # Fetch the maximum number of pages per request (usually 500)
    }
    
    all_players = []
    continue_token = None

    # Pagination
    while True:
        if continue_token:
            params['cmcontinue'] = continue_token

        response = requests.get(url, headers=headers, params=params)
        data = response.json()

        all_players.extend(data["query"]["categorymembers"])

        # Check if there's more data to fetch
        if "continue" in data:
            continue_token = data["continue"]["cmcontinue"]
        else:
            break

    return all_players

In [13]:
players = get_player_pages(wiki="leagueoflegends")

In [14]:
len(players)

4574

In [17]:
players[0]

{'pageid': 26133, 'ns': 0, 'title': '007x'}

In [27]:
player_names = [player["title"] for player in players]

In [23]:
import re

In [55]:
def find_matching_items(items, pattern):
    """Pre-processing step to strip the extra brackets stuff which is usually used to distinguish players with the same name (which we don't want!)."""
    # Use list comprehension to filter items that match the pattern
    matching_items = [item for item in items if regex.search(item)]
    
    return matching_items

In [56]:
test_list = ["A((el", "A((el (Korean player)", "A(b", "()", "( B )"]

In [57]:
# i think i like the regex of at least 1 letter, space, left bracket, at least 1 letter, right bracket, ENDS
# if it matches that, remove everything from the far right left bracket
# JUST in case there's usernames with brackets in them
pattern = r".+\ \(.+\)$"
# Compile the regex pattern for efficiency
regex = re.compile(pattern)

matches = find_matching_items(player_names, pattern)
print(matches)
print(len(matches))

['Ace (Japanese player)', 'Ace (Korean player)', 'Ackerman (Argentinian player)', 'Ady (Korean player)', 'Air (Chinese player)', 'Akashi (Moroccan player)', 'Aki (Italian player)', 'Alvaro (Álvaro Fernández)', 'Amazing (German player)', 'Angel (Chinese player)', 'Archie (Vietnamese player)', 'Art (Taiwanese caster)', 'Artemis (American player)', 'Arthur (Park Mi-reu)', 'Asta (Australian player)', 'Asta (Brazilian player)', 'Atom (Danish player)', 'Azure (Filipino player)', 'Bankai (Renan Pirone)', 'Benny (Taiwanese player)', 'Berserker (Korean player)', 'Beyond (Vietnamese player)', 'Billy (Taiwanese player)', 'Bless (Chinese player)', 'Bless (Korean player)', 'Blue (Australian player)', 'Bobo (Austrian player)', 'Bohe (Wang Tiannuo)', 'BOSS (Russian player)', 'BRUCE (Japanese player)', 'Brush (Oh Jun-young)', 'Bung (Austrian player)', 'Calix (Syun Hyun-bin)', 'Calm (Vietnamese player)', 'Candy (Korean player)', 'Carnage (Greek player)', 'Carry (Turkish player)', 'Casper (Jordanian pla

In [61]:
def isolate_username(player_name):
    if regex.search(player_name):
        # strip everything from the last space + left bracket
        index = player_name.rfind(" (")
    
        # If a left bracket is found, slice the string up to and including this index
        if index != -1:
            return player_name[:index]
        else:
            raise ValueException("Regex pattern must be wrong")
            # return player_name
    else:
        return player_name

In [63]:
cleaned_player_names = [isolate_username(player_name) for player_name in player_names]

In [18]:
from collections import Counter

In [64]:
value_counts = Counter([player_name for player_name in cleaned_player_names])

In [65]:
value_counts.most_common()

Counter({'Nova': 4,
         'Feng': 3,
         'Hiro': 3,
         'Jay': 3,
         'Moo': 3,
         'River': 3,
         'Wolf': 3,
         'Woong': 3,
         'Yuki': 3,
         'Ace': 2,
         'Art': 2,
         'Artemis': 2,
         'Asta': 2,
         'Beyond': 2,
         'Bless': 2,
         'Blue': 2,
         'Castle': 2,
         'Chelly': 2,
         'Chico': 2,
         'Cloud': 2,
         'Crazy': 2,
         'Crown': 2,
         'Cube': 2,
         'Dawn': 2,
         'Decoy': 2,
         'Demon': 2,
         'Dice': 2,
         'Djoko': 2,
         'Dragon': 2,
         'Duke': 2,
         'Flare': 2,
         'Fly': 2,
         'Frozen': 2,
         'Hades': 2,
         'Helper': 2,
         'Hoon': 2,
         'Hyper': 2,
         'Icarus': 2,
         'Jun': 2,
         'Kaze': 2,
         'Kitty': 2,
         'Leon': 2,
         'Lucky': 2,
         'Lunar': 2,
         'Lustboy': 2,
         'Mac': 2,
         'Maestro': 2,
         'Magic': 2,
       

There are indeed 4 'notable players/coaches/personalities' with the pseudonym Nova...

1. https://liquipedia.net/leagueoflegends/Nova
2. https://liquipedia.net/leagueoflegends/Nova_(Jonathan_Baadsgaard)
3. https://liquipedia.net/leagueoflegends/Nova_(Moroccan_player)
4. https://liquipedia.net/leagueoflegends/Nova_(Park_Chan-ho)

And that's lol alone.